# ODI to Databricks Migration: TRG_LOC Full Load**Source File:** `TRG_LOC.txt`**Conversion Timestamp:** 2024-07-30 12:00:00 UTCThis notebook performs a full load of location data into `workspace.hr.trg_loc`.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "1. ETL Job Type (e.g., 'F' for Full Load)")
dbutils.widgets.text("DATASOURCE_NUM_ID", "1", "2. Data Source Number ID")
dbutils.widgets.text("ETL_PROC_WID", "-1", "3. ETL Process Widget ID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "4. ODI Session Number")

# ETL Parameters

In [ ]:
display(spark.sql("""
  SELECT
    '${ETL_JOB_TYPE}' AS ETL_JOB_TYPE,
    ${DATASOURCE_NUM_ID} AS DATASOURCE_NUM_ID,
    ${ETL_PROC_WID} AS ETL_PROC_WID,
    '${ODI_SESS_NO}' AS ODI_SESS_NO
"""))

# Target Table: TRG_LOC

In [ ]:
%sql
-- SCEN_TASK_NO in {10}: Drop target table (Logical drop/recreate for full load)
DROP TABLE IF EXISTS workspace.hr.trg_loc;

In [ ]:
%sql
-- SCEN_TASK_NO in {20}: Create target table (Inferred schema from source LOCATIONS)
CREATE TABLE workspace.hr.trg_loc (
    LOCATION_ID     BIGINT,
    STREET_ADDRESS  STRING,
    POSTAL_CODE     STRING,
    CITY            STRING,
    STATE_PROVINCE  STRING,
    COUNTRY_ID      STRING
) USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO in {30}: Insert data into TRG_LOC
INSERT INTO workspace.hr.trg_loc (
    LOCATION_ID,
    STREET_ADDRESS,
    POSTAL_CODE,
    CITY,
    STATE_PROVINCE,
    COUNTRY_ID
)
SELECT
    LOCATIONS.LOCATION_ID,
    LOCATIONS.STREET_ADDRESS,
    LOCATIONS.POSTAL_CODE,
    LOCATIONS.CITY,
    LOCATIONS.STATE_PROVINCE,
    LOCATIONS.COUNTRY_ID
FROM
    workspace.hr.locations AS LOCATIONS;

In [ ]:
%sql
SELECT COUNT(*) AS record_count FROM workspace.hr.trg_loc;

# Optimize Target

In [ ]:
%sql
-- Disable ZORDER stats check to prevent DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.hr.trg_loc ZORDER BY (LOCATION_ID);

# Validation

In [ ]:
%sql
SELECT * FROM workspace.hr.trg_loc LIMIT 10;

# Conversion Notes and Manual Actions Required1.  **Schema Inference**: The `CREATE TABLE` statement for `workspace.hr.trg_loc` has been inferred based on common Oracle HR schema data types. Please verify the inferred types (`BIGINT`, `STRING`) against your actual source `HR.LOCATIONS` table DDL to ensure accuracy and adjust if necessary.2.  **Missing ODI Tasks**: `SCEN_TASK_NO in {10}, {20}, {30}` were present in the source but did not contain any explicit SQL statements in the provided snippet beyond the `INSERT` itself. I've placed the `DROP TABLE`, `CREATE TABLE`, and `INSERT` commands under these markers respectively, assuming this is the intended execution flow for a full load.3.  **No Widgets Used**: This specific ODI snippet did not contain any ODI parameters (e.g., `#GLOBAL.param`). However, standard ETL widgets (`ETL_JOB_TYPE`, `DATASOURCE_NUM_ID`, `ETL_PROC_WID`, `ODI_SESS_NO`) have been included in the notebook as per the system prompt for completeness.